## Data Preprocessing - Time

In [1]:
import numpy as np
import pandas as pd
import datetime
from datetime import datetime, timedelta
import holidays

# import matplotlib
from matplotlib import pyplot
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import matplotlib.image as mpimg

# import seaborn
import seaborn as sns

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [2]:
# to be able to see multiple outputs from single cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

#### Import data

In [3]:
processed_df = pd.read_csv('../../data/processed/final_for_EDA/ttc_cleaned_final.csv')

In [4]:
processed_df.info()
processed_df

<class 'pandas.DataFrame'>
RangeIndex: 20316 entries, 0 to 20315
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   year       20316 non-null  int64
 1   month      20316 non-null  str  
 2   date       20316 non-null  str  
 3   time       20316 non-null  str  
 4   day        20316 non-null  str  
 5   station    20316 non-null  str  
 6   code       20316 non-null  str  
 7   min_delay  20316 non-null  int64
 8   min_gap    20316 non-null  int64
 9   bound      20316 non-null  str  
 10  line       20316 non-null  str  
 11  vehicle    20316 non-null  int64
dtypes: int64(4), str(8)
memory usage: 1.9 MB


,year,month,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle
0,2025,March,2025-03-16,13:49,Sunday,old mill,PUTIJ,10,14,E,BD,5103
1,2025,January,2025-01-01,2:10,Wednesday,bathurst,MUSAN,5,9,E,BD,5227
2,2025,February,2025-02-05,20:38,Wednesday,bathurst,SUDP,5,9,W,BD,5148
3,2025,March,2025-03-21,11:40,Friday,bathurst,MUDD,5,9,W,BD,5170
4,2025,June,2025-06-11,15:32,Wednesday,bathurst,SUO,5,9,E,BD,5035
...,...,...,...,...,...,...,...,...,...,...,...,...
20311,2021,July,2021-07-02,22:20,Friday,yorkdale,SUDP,5,12,N,YU,5691
20312,2021,July,2021-07-27,18:04,Tuesday,yorkdale,TUO,5,8,S,YU,5431
20313,2021,August,2021-08-08,18:23,Sunday,yorkdale,MUO,5,10,S,YU,5856
20314,2026,January,2026-01-25,17:23,Sunday,yorkdale,PUTIS,5,10,N,YU,6056


#### Convert Time column in string to time data type

In [5]:
t_adj = pd.to_datetime(processed_df['time'].astype(str).str.strip(), errors="coerce")

processed_df['time_hms'] = t_adj.dt.time

In [9]:
# function to convert time column in time data type to time categories
def categorize_time_of_day(time_val):

    hour = time_val.hour

    if (hour >= 0) & (hour < 6):
        return "Early_AM"
    elif (hour >= 6) & (hour < 9):
        return "AM_Peak"
    elif (hour >= 9) & (hour < 15):
        return "Mid_Day"
    elif (hour >= 15) & (hour < 19):
        return "PM_Peak"
    elif (hour >= 19) & (hour < 22):
        return "Evening"
    elif (hour >= 22) & (hour < 24):
        return "Late_Evening"
    else:
        return np.nan
    

# function to convert day to weekday, weekend and holidays
def convert_weekday_weekend_holiday(day_val, date_val):
    
    ca_holidays = holidays.Canada()
    date = pd.to_datetime(date_val).date()
    
    if date in ca_holidays:
        return "Holiday" 
    elif day_val in ("Saturday", "Sunday"):
        return "Weekend" 
    elif day_val in ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday"):
        return "Weekday" 
    else:
        return "Unknown"

In [10]:
processed_df["time_category"] = processed_df["time_hms"].apply(categorize_time_of_day)
processed_df["day_category"] = processed_df.apply(lambda x: convert_weekday_weekend_holiday(x.day, x.date), axis=1)

In [11]:
processed_df.info()
processed_df

<class 'pandas.DataFrame'>
RangeIndex: 20316 entries, 0 to 20315
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   year           20316 non-null  int64 
 1   month          20316 non-null  str   
 2   date           20316 non-null  str   
 3   time           20316 non-null  str   
 4   day            20316 non-null  str   
 5   station        20316 non-null  str   
 6   code           20316 non-null  str   
 7   min_delay      20316 non-null  int64 
 8   min_gap        20316 non-null  int64 
 9   bound          20316 non-null  str   
 10  line           20316 non-null  str   
 11  vehicle        20316 non-null  int64 
 12  time_hms       20316 non-null  object
 13  time_category  20316 non-null  str   
 14  day_category   20316 non-null  str   
dtypes: int64(4), object(1), str(10)
memory usage: 2.3+ MB


,year,month,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle,time_hms,time_category,day_category
0,2025,March,2025-03-16,13:49,Sunday,old mill,PUTIJ,10,14,E,BD,5103,13:49:00,Mid_Day,Weekend
1,2025,January,2025-01-01,2:10,Wednesday,bathurst,MUSAN,5,9,E,BD,5227,02:10:00,Early_AM,Holiday
2,2025,February,2025-02-05,20:38,Wednesday,bathurst,SUDP,5,9,W,BD,5148,20:38:00,Evening,Weekday
3,2025,March,2025-03-21,11:40,Friday,bathurst,MUDD,5,9,W,BD,5170,11:40:00,Mid_Day,Weekday
4,2025,June,2025-06-11,15:32,Wednesday,bathurst,SUO,5,9,E,BD,5035,15:32:00,PM_Peak,Weekday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20311,2021,July,2021-07-02,22:20,Friday,yorkdale,SUDP,5,12,N,YU,5691,22:20:00,Late_Evening,Weekday
20312,2021,July,2021-07-27,18:04,Tuesday,yorkdale,TUO,5,8,S,YU,5431,18:04:00,PM_Peak,Weekday
20313,2021,August,2021-08-08,18:23,Sunday,yorkdale,MUO,5,10,S,YU,5856,18:23:00,PM_Peak,Weekend
20314,2026,January,2026-01-25,17:23,Sunday,yorkdale,PUTIS,5,10,N,YU,6056,17:23:00,PM_Peak,Weekend


In [12]:
processed_df.to_csv('../../data/processed/final_for_EDA/ttc_cleaned_final_with_peaktimes.csv', index=False)